<a href="https://colab.research.google.com/github/Manish927/EDA-Data-Science/blob/main/neural_machine_translation_en_hi_lstm_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import tensorflow as tf
import numpy as np
import pandas as pd
import re
from datasets import load_dataset
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

In [59]:
data = [
("i am going to the market to buy fruits","मैं बाजार जा रहा हूँ फल खरीदने के लिए"),
("she is reading a book in the quiet library","वह शांत पुस्तकालय में एक किताब पढ़ रही है"),
("we are planning a trip to the mountains next week","हम अगले सप्ताह पहाड़ों की यात्रा की योजना बना रहे हैं"),
("he is working on a project for his college assignment","वह अपने कॉलेज असाइनमेंट के लिए एक परियोजना पर काम कर रहा है"),
("they are watching a movie together at home tonight","वे आज रात घर पर एक साथ फिल्म देख रहे हैं"),
("the teacher is explaining the topic very clearly to students","शिक्षक छात्रों को विषय बहुत स्पष्ट रूप से समझा रहे हैं"),
("i like to play cricket with my friends in the evening","मुझे शाम को अपने दोस्तों के साथ क्रिकेट खेलना पसंद है"),
("she cooked a delicious meal for her family yesterday evening","उसने कल शाम अपने परिवार के लिए स्वादिष्ट भोजन बनाया"),
("we should complete the work before the deadline tomorrow morning","हमें कल सुबह समय सीमा से पहले काम पूरा करना चाहिए"),
("he bought a new laptop for learning programming and data science","उसने प्रोग्रामिंग और डेटा साइंस सीखने के लिए नया लैपटॉप खरीदा"),

# NEW DATA ↓↓↓
("i am going home after buying fruits from the market","मैं बाजार से फल खरीदकर घर जा रहा हूँ"),
("she is writing notes for her upcoming exams in the room","वह अपने आने वाली परीक्षा के लिए कमरे में नोट्स लिख रही है"),
("we are playing football in the ground with our friends","हम अपने दोस्तों के साथ मैदान में फुटबॉल खेल रहे हैं"),
("he is cooking dinner for his family in the kitchen","वह रसोई में अपने परिवार के लिए खाना बना रहा है"),
("they are going to attend a wedding ceremony this evening","वे आज शाम एक शादी समारोह में जा रहे हैं"),
("the students are studying hard for their final examinations","छात्र अपनी अंतिम परीक्षाओं के लिए कड़ी मेहनत कर रहे हैं"),
("i am learning machine learning and deep learning concepts","मैं मशीन लर्निंग और डीप लर्निंग के सिद्धांत सीख रहा हूँ"),
("she is practicing yoga every morning for better health","वह बेहतर स्वास्थ्य के लिए हर सुबह योग का अभ्यास करती है"),
("we are discussing the project requirements in the meeting","हम बैठक में परियोजना की आवश्यकताओं पर चर्चा कर रहे हैं"),
("he is traveling to another city for his office work","वह अपने कार्यालय के काम के लिए दूसरे शहर जा रहा है"),

("i am reading a newspaper while drinking tea in the morning","मैं सुबह चाय पीते हुए अखबार पढ़ रहा हूँ"),
("she is cleaning the house before the guests arrive today","वह आज मेहमानों के आने से पहले घर साफ कर रही है"),
("we are watching a cricket match on television together","हम साथ में टेलीविजन पर क्रिकेट मैच देख रहे हैं"),
("he is fixing his car in the garage with some tools","वह कुछ औजारों के साथ गैरेज में अपनी कार ठीक कर रहा है"),
("they are preparing food for the festival celebration tonight","वे आज रात त्योहार के लिए खाना तैयार कर रहे हैं"),
("the manager is reviewing the reports carefully before submission","प्रबंधक जमा करने से पहले रिपोर्ट को ध्यान से देख रहा है"),
("i am working on improving my coding skills every day","मैं हर दिन अपनी कोडिंग कौशल सुधारने पर काम कर रहा हूँ"),
("she is talking to her friend on the phone right now","वह अभी अपने दोस्त से फोन पर बात कर रही है"),
("we are waiting for the bus at the station since morning","हम सुबह से स्टेशन पर बस का इंतजार कर रहे हैं"),
("he is listening to music while doing his office work","वह अपना ऑफिस का काम करते हुए संगीत सुन रहा है"),

("i am going to attend an important meeting in the office","मैं कार्यालय में एक महत्वपूर्ण बैठक में जाने वाला हूँ"),
("she is preparing a presentation for her college project","वह अपने कॉलेज प्रोजेक्ट के लिए प्रस्तुति तैयार कर रही है"),
("we are learning new technologies to improve our career opportunities","हम अपने करियर के अवसरों को बेहतर बनाने के लिए नई तकनीक सीख रहे हैं"),
("he is teaching students in the classroom with great enthusiasm","वह कक्षा में छात्रों को बड़े उत्साह के साथ पढ़ा रहा है"),
("they are enjoying their vacation in a beautiful hill station","वे एक सुंदर पहाड़ी स्थान में अपनी छुट्टियों का आनंद ले रहे हैं"),
("the doctor is checking patients in the hospital regularly","डॉक्टर अस्पताल में मरीजों की नियमित जांच कर रहा है"),
("i am planning to start a new project related to artificial intelligence","मैं कृत्रिम बुद्धिमत्ता से संबंधित एक नया प्रोजेक्ट शुरू करने की योजना बना रहा हूँ"),
("she is designing a new dress for the upcoming fashion show","वह आने वाले फैशन शो के लिए एक नई ड्रेस डिजाइन कर रही है"),
("we are organizing an event for our company employees next week","हम अगले सप्ताह अपने कंपनी कर्मचारियों के लिए एक कार्यक्रम आयोजित कर रहे हैं"),
("he is building a new application using modern software technologies","वह आधुनिक सॉफ्टवेयर तकनीकों का उपयोग करके एक नया एप्लिकेशन बना रहा है")
]
eng_sentences = [x[0] for x in data]
hin_sentences = ["start " + x[1] + " end" for x in data]

print(eng_sentences)
print(hin_sentences)

['i am going to the market to buy fruits', 'she is reading a book in the quiet library', 'we are planning a trip to the mountains next week', 'he is working on a project for his college assignment', 'they are watching a movie together at home tonight', 'the teacher is explaining the topic very clearly to students', 'i like to play cricket with my friends in the evening', 'she cooked a delicious meal for her family yesterday evening', 'we should complete the work before the deadline tomorrow morning', 'he bought a new laptop for learning programming and data science', 'i am going home after buying fruits from the market', 'she is writing notes for her upcoming exams in the room', 'we are playing football in the ground with our friends', 'he is cooking dinner for his family in the kitchen', 'they are going to attend a wedding ceremony this evening', 'the students are studying hard for their final examinations', 'i am learning machine learning and deep learning concepts', 'she is practici

In [60]:
# Tokenization

eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(eng_sentences)
eng_seq = eng_tokenizer.texts_to_sequences(eng_sentences)

hin_tokenizer = Tokenizer(filters="")
hin_tokenizer.fit_on_texts(hin_sentences)
hin_seq = hin_tokenizer.texts_to_sequences(hin_sentences)

eng_vocab_size = len(eng_tokenizer.word_index) + 1
hin_vocab_size = len(hin_tokenizer.word_index) + 1

In [61]:
# Padding

max_eng_len = max(len(seq) for seq in eng_seq)
max_hin_len = max(len(seq) for seq in hin_seq)

encoder_input = pad_sequences(eng_seq, maxlen=max_eng_len, padding='post')
decoder_input = pad_sequences(hin_seq, maxlen=max_hin_len, padding='post')

decoder_target = np.zeros_like(decoder_input)
decoder_target[:, :-1] = decoder_input[:, 1:]

In [62]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model
import tensorflow as tf

# Encoder
encoder_inputs = Input(shape=(max_eng_len,))
encoder_embedding = Embedding(eng_vocab_size, latent_dim)
enc_emb = encoder_embedding(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_hin_len,))
decoder_embedding = Embedding(hin_vocab_size, latent_dim)
dec_emb = decoder_embedding(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(hin_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

Model: "functional_25"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_53      │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_54      │ (None, 17)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_16        │ (None, 12, 128)   │     24,320 │ input_layer_53[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_17        │ (None, 17, 128)   │     25,600 │ input_layer_54[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_16 (LSTM)      │ [(None, 128),     │    131,584 │ embedding_16[0][… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_17 (LSTM)      │ [(None, 17, 128), │    131,584 │ embedding_17[0][… │
│                     │ (None, 128),      │            │ lstm_16[0][1],    │
│                     │ (None, 128)]      │            │ lstm_16[0][2]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 17, 200)   │     25,800 │ lstm_17[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 338,888 (1.29 MB)

 Trainable params: 338,888 (1.29 MB)

 Non-trainable params: 0 (0.00 B)

In [63]:
model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=2,
    epochs=500,
    verbose=1
)

Epoch 1/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.2529 - loss: 4.8095
Epoch 2/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2691 - loss: 3.7785
Epoch 3/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2809 - loss: 3.4937
Epoch 4/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3132 - loss: 3.3663
Epoch 5/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3529 - loss: 3.2016
Epoch 6/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3721 - loss: 3.0263
Epoch 7/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3838 - loss: 2.9336
Epoch 8/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3926 - loss: 2.8553
Epoch 9/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4029 - loss: 2.7752
Epoch 10/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4250 - loss: 2.6808
Epoch 11/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4515 - loss: 2.6160
Epoch 12/500
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - ac

In [64]:
# Inference (Actual Translation)
encoder_model = Model(encoder_inputs, encoder_states)

In [65]:
# Decoder setup (clean version)

# Encoder model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_single_input = Input(shape=(1,))

dec_emb2 = decoder_embedding(decoder_single_input)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)

decoder_states2 = [state_h2, state_c2]

decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model(
    [decoder_single_input] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

In [66]:
reverse_hin_index = {v: k for k, v in hin_tokenizer.word_index.items()}

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = hin_tokenizer.word_index['start']

    decoded_sentence = ""

    for _ in range(max_hin_len):
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_hin_index.get(sampled_token_index, "")

        if sampled_word == "end":
            break

        decoded_sentence += " " + sampled_word

        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return decoded_sentence.strip()

In [67]:
def translate(sentence):
    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_eng_len, padding='post')
    return decode_sequence(seq)

print(translate("i am going to the market to buy fruits"))

print(translate("i am fine"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
मैं बाजार जा रहा हूँ फल खरीदने के लिए
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
हम अपने दोस्तों के साथ मैदान में फुटबॉल खेल रहे 

### This is the core limitation of your current model. Encoder reads entire sentence -> compresses into one fixed vector. Decoder uses that vector to generate all words.

### Issues: for long sentences one vector cannot store subject, verb, object, context. Information gets lost

# Attention
### Instead of one vector. Decoder looks at all encoder outputs at every step.
While translating:

When generating "बाजार"
→ model focuses on "market"
When generating "फल"
→ model focuses on "fruits"


### Sentence → [many vectors]
                 ↓
Decoder attends to relevant words dynamically

In [68]:
from tensorflow.keras.layers import Attention

In [69]:
# Modify Model (ONLY decoder part changes) MODEL with attention

from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Attention, Concatenate
from tensorflow.keras.models import Model
import tensorflow as tf

latent_dim = 128

# Encoder
encoder_inputs = Input(shape=(max_eng_len,))
encoder_embedding = Embedding(eng_vocab_size, latent_dim)
enc_emb = encoder_embedding(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_hin_len,))
decoder_embedding = Embedding(hin_vocab_size, latent_dim)
dec_emb = decoder_embedding(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

# Attention
attention_layer = Attention()
attention_output = attention_layer([decoder_outputs, encoder_outputs])

# Concatenate
concat_layer = Concatenate(axis=-1)
decoder_concat = concat_layer([decoder_outputs, attention_output])

# Output
decoder_dense = Dense(hin_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_concat)

# Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

model.summary()

Model: "functional_29"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_58      │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_59      │ (None, 17)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_18        │ (None, 12, 128)   │     24,320 │ input_layer_58[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_19        │ (None, 17, 128)   │     25,600 │ input_layer_59[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_18 (LSTM)      │ [(None, 12, 128), │    131,584 │ embedding_18[0][… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_19 (LSTM)      │ [(None, 17, 128), │    131,584 │ embedding_19[0][… │
│                     │ (None, 128),      │            │ lstm_18[0][1],    │
│                     │ (None, 128)]      │            │ lstm_18[0][2]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_6         │ (None, 17, 128)   │          0 │ lstm_19[0][0],    │
│ (Attention)         │                   │            │ lstm_18[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 17, 256)   │          0 │ lstm_19[0][0],    │
│ (Concatenate)       │                   │            │ attention_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 17, 200)   │     51,400 │ concatenate_5[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 364,488 (1.39 MB)

 Trainable params: 364,488 (1.39 MB)

 Non-trainable params: 0 (0.00 B)

In [70]:
model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    batch_size=2,
    epochs=500,
    verbose=0
)

In [71]:
# Encoder model, Inference model
encoder_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

# Decoder inputs
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
encoder_outputs_input = Input(shape=(max_eng_len, latent_dim))

decoder_single_input = Input(shape=(1,))

# Embedding
dec_emb2 = decoder_embedding(decoder_single_input)

# LSTM
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=[decoder_state_input_h, decoder_state_input_c]
)

# Attention
attention_out = attention_layer([decoder_outputs2, encoder_outputs_input])

# Concatenate
decoder_concat2 = concat_layer([decoder_outputs2, attention_out])

# Dense
decoder_outputs2 = decoder_dense(decoder_concat2)

# Final decoder model
decoder_model = Model(
    [decoder_single_input, encoder_outputs_input, decoder_state_input_h, decoder_state_input_c],
    [decoder_outputs2, state_h2, state_c2]
)

In [72]:
# Decode Function

reverse_hin_index = {v: k for k, v in hin_tokenizer.word_index.items()}

def decode_sequence(input_seq):
    encoder_out, state_h, state_c = encoder_model.predict(input_seq)

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = hin_tokenizer.word_index['start']

    decoded_sentence = ""

    for _ in range(max_hin_len):
        output_tokens, h, c = decoder_model.predict(
            [target_seq, encoder_out, state_h, state_c]
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_hin_index.get(sampled_token_index, "")

        if sampled_word == "end":
            break

        decoded_sentence += " " + sampled_word

        target_seq[0, 0] = sampled_token_index
        state_h, state_c = h, c

    return decoded_sentence.strip()

In [75]:
# Translation

def translate(sentence):
    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_eng_len, padding='post')
    return decode_sequence(seq)

print(translate("i am going to the market to buy fruits"))

print(translate("i am learning machine learning and deep learning concepts"))

print(translate("i am going to attend machine learning"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
मैं बाजार जा रहा हूँ फल खरीदने के लिए
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
मैं मशीन लर्निंग और डीप लर्निंग के सिद्धांत सीख रह